In [ ]:
"""
Fine-tunes an ImageNet-pretrained ResNet-50 to classify skin lesions as malignant (1) or benign (0).
Uses the training and validation splits from prep_data.py
"""
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import json
import random
import time
from pathlib import Path

from lesion_dataset import LesionDataset
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader
from torchvision import models, transforms

# ---------------- Settings ----------------
SPLITS = Path("splits") # folder with train.csv and val.csv from prep_data.py
SEED = 3 # change to 1, 2, or 3 for each run
OUT_DIR = Path(f"runs/seed{SEED}") # seed saved to its folder
EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-4
IMAGE_SIZE = 224
NUM_WORKERS = 4
LIMIT = 0 # use only 500 training images while testing
TARGET_SPECIFICITY = 0.78 # human readers' specificity (Tschandl et al., 2019)
MALIGNANT_TYPES = ["mel", "bcc", "akiec"]
# average of ImageNet's red, green, and blue values.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [ ]:
# apply random crops, flips, and rotations for training
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# images stay the same during validation
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
def pick_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def build_model():
    """ResNet-50 with ImageNet weights and a 1 output last layer."""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """A full pass through the training images. Return the average loss."""
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze(1)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset)

In [ ]:
def predict(model, loader, device):
    """Scores every image in the loader. Return the probabilities."""
    model.eval()
    all_probs = []

    with torch.no_grad():
        for images, _ in loader:
            outputs = model(images.to(device)).squeeze(1)
            all_probs.append(torch.sigmoid(outputs).cpu())
            
    return torch.cat(all_probs).numpy()


def sens_spec(labels, probs, cutoff):
    """Sensitivity and specificity when scores at or above the cutoff are called malignant. Return a value between 0 and 1 for each."""
    predicted_malignant = probs >= cutoff
    is_malignant = labels == 1
    is_benign = labels == 0
    sensitivity = (predicted_malignant & is_malignant).sum() / is_malignant.sum()
    specificity = (~predicted_malignant & is_benign).sum() / is_benign.sum()
    return float(sensitivity), float(specificity)


def cutoff_for_specificity(labels, probs, target):
    """Return the lowest cutoff that keeps specificity at or above the target."""
    fpr, tpr, thresholds = roc_curve(labels, probs)
    meets_target = np.where(1 - fpr >= target)[0]
    best_index = meets_target[-1]
    return float(min(thresholds[best_index], 1.0))


def compute_metrics(val_df, probs):
    """Return a dictionary of validation metrics.
    Includes AUROC, the cutoff that gives the target specificity, sensitivity and specificity at that cutoff, and sensitivity separately for each malignant type."""
    labels = val_df["label"].to_numpy()
    auroc = roc_auc_score(labels, probs)
    cutoff = cutoff_for_specificity(labels, probs, TARGET_SPECIFICITY)
    sensitivity, specificity = sens_spec(labels, probs, cutoff)
    by_type = {}
    for dx in MALIGNANT_TYPES:
        mask = (val_df["dx"] == dx).to_numpy()
        if mask.any():
            by_type[dx] = float((probs[mask] >= cutoff).mean())
    return {"auroc": float(auroc), "cutoff": cutoff, "sensitivity": sensitivity, "specificity": specificity, "sensitivity_by_type": by_type}

In [ ]:
def main():
    # fix the random generators
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # load the split from prep_data.py
    train_df = pd.read_csv(SPLITS / "train.csv")
    val_df = pd.read_csv(SPLITS / "val.csv")

    # if a limit is set for the small subset
    if LIMIT:
        train_df = train_df.sample(n=min(LIMIT, len(train_df)), random_state=SEED)
        val_df = val_df.sample(n=min(LIMIT // 4, len(val_df)), random_state=SEED)
    val_df = val_df.reset_index(drop=True) # renumber rows so that they match their position

    # class counts for the weighted loss
    n_pos = int(train_df["label"].sum())
    n_neg = len(train_df) - n_pos
    print(f"Train: {len(train_df)} images ({n_pos} malignant), Val: {len(val_df)} images")

    # provides the batches
    train_loader = DataLoader(LesionDataset(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS) # training shuffles each epoch
    val_loader = DataLoader(LesionDataset(val_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS) # validation keeps fixed order
    
    device = pick_device()
    print("Device:", device)
    model = build_model().to(device)

    # calculate weighted loss for class imbalance
    # missed malignant case costs n_neg / n_pos (about 4) times more than a false alarm
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4) # AdamW updates all weights
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS) # lowers the learning rate over each epoch
    
    history = [] # holds metrics per epoch
    best = None # record the best epoch
    # training loop
    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        scheduler.step()
        probs = predict(model, val_loader, device) # score validation images
        m = compute_metrics(val_df, probs) # current epoch's AUROC, cutoff, sensitivity, specificity
        minutes = (time.time() - start) / 60
        print(f"Epoch {epoch}/{EPOCHS}  loss {train_loss:.3f}  AUROC {m['auroc']:.3f} sens {m['sensitivity']:.3f} @ spec {m['specificity']:.3f}  ({minutes:.1f} min)")

        # save progress every epoch 
        history.append({"epoch": epoch, "train_loss": train_loss, "auroc": m["auroc"], "sensitivity": m["sensitivity"], "specificity": m["specificity"], "minutes": minutes})
        pd.DataFrame(history).to_csv(OUT_DIR / "history.csv", index=False)

        # keep the epoch with the best validation AUROC
        if best is None or m["auroc"] > best["auroc"]:
            best = {**m, "epoch": epoch}
            torch.save({"model": model.state_dict(), "cutoff": m["cutoff"]}, OUT_DIR / "best_model.pt")
            preds = val_df[["image_id", "dx", "label"]].copy()
            preds["prob_malignant"] = probs
            preds.to_csv(OUT_DIR / "val_predictions.csv", index=False)

    # metrics of the best epoch
    (OUT_DIR / "metrics.json").write_text(json.dumps(best, indent=2))
    print(f"\nBest epoch {best['epoch']}: AUROC {best['auroc']:.3f}, sensitivity {best['sensitivity']:.3f} at specificity {best['specificity']:.3f}")
    print("Sensitivity by type:", best["sensitivity_by_type"])

if __name__ == "__main__":
    main()

Train: 8584 images (1675 malignant), Val: 1431 images
Device: mps
Epoch 1/10  loss 0.695  AUROC 0.898  sens 0.846 @ spec 0.784  (2.7 min)
Epoch 2/10  loss 0.532  AUROC 0.908  sens 0.864 @ spec 0.794  (2.6 min)
Epoch 3/10  loss 0.464  AUROC 0.918  sens 0.892 @ spec 0.782  (2.6 min)
Epoch 4/10  loss 0.399  AUROC 0.919  sens 0.889 @ spec 0.792  (2.6 min)
Epoch 5/10  loss 0.345  AUROC 0.913  sens 0.867 @ spec 0.781  (2.6 min)
Epoch 6/10  loss 0.280  AUROC 0.919  sens 0.903 @ spec 0.781  (2.6 min)
Epoch 7/10  loss 0.222  AUROC 0.925  sens 0.925 @ spec 0.782  (2.7 min)
Epoch 8/10  loss 0.186  AUROC 0.932  sens 0.918 @ spec 0.783  (2.7 min)
Epoch 9/10  loss 0.149  AUROC 0.929  sens 0.925 @ spec 0.781  (2.7 min)
Epoch 10/10  loss 0.141  AUROC 0.930  sens 0.918 @ spec 0.784  (2.7 min)

Best epoch 8: AUROC 0.932, sensitivity 0.918 at specificity 0.783
Sensitivity by type: {'mel': 0.8809523809523809, 'bcc': 1.0, 'akiec': 0.9333333333333333}
